***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
import geopandas as gpd
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.max_columns', None)

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'Zillow')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Zillow Data')
    path_main = os.path.join(path_sp, 'Data')

    
path_config0 = os.path.join(path_git, 'config')
path_code    = os.path.join(path_git, 'Data', 'Zillow')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

In [ ]:
# Assign geographies
sacog_state = ["Sacramento, CA", "Yuba City, CA"]

ca_peers = ["Los Angeles, CA", "San Francisco, CA", "Riverside, CA", 
            "San Diego, CA", "Oxnard, CA", "Santa Rosa, CA", 
            "Vallejo, CA", "El Centro, CA", "Napa, CA", "San Jose, CA"]

other_peers = ["Austin, TX", "Charlotte, NC", "Cincinnati, OH",
               "Cleveland, OH", "Columbus, OH", "Detroit, MI",
               "Indianapolis, IN", "Kansas City, KS", "Miami, FL",
               "Orlando, FL", "Phoenix, AZ", "Pittsburg, PA",
               "Portland, OR", "Salt Lake City, UT", "San Antonio, TX",
               "St. Louis, MO", "Tampa, FL"]

# More classifiers
sacog = ["Yuba City", "Sacramento"]
mtc   = ["San Francisco", "Santa Rosa", "Vallejo", "Napa", 'San Jose']

scag  = ["Los Angeles", "Riverside", "Oxnard", "El Centro"]

In [ ]:
sample_type = 'Zillow'
year_start = 2000
year_end = 2023
geography = 'MPO'

***

Cost_1

***

In [ ]:
indicator_name = 'Cost_1'

df_about = write_about(sample_type=sample_type
                       , indicator_name=indicator_name
                       , year_start=year_start
                       , year_end=year_end
                       , path_config0=path_config0
                       , geography=geography)
print("About page documentation table:")
display(df_about)
print('')

# Load CSV file
df_cost1 = pd.read_csv(os.path.join(path_raw, "Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"))
df_cost1 = df_cost1[df_cost1['RegionType'] == 'msa']
df_cost1 = pd.melt(df_cost1, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'], var_name='date_', value_name='Price')

# Subset and rename cols
df_cost1 = df_cost1[['RegionName', 'StateName', 'date_', 'Price']]
df_cost1.columns = ['Region', 'State', 'date_', 'Price']
df_cost1 = df_cost1[df_cost1['Region'].isin(sacog_state + ca_peers + other_peers)]

# Removing trailing text from Region
df_cost1['Region'] = df_cost1['Region'].str.replace(r',.*', '', regex=True)

# Mutate function to get MPO col
df_cost1['MPO'] = np.where(df_cost1['Region'].isin(sacog), 'SACOG'
                , np.where(df_cost1['Region'].isin(mtc  ), 'MTC'
                , np.where(df_cost1['Region'].isin(scag ), 'SCAG'
                , np.where(df_cost1['Region'] == 'San Diego', 'SANDAG', df_cost1['Region']))))

# Calculate median prices for each MPO

sacog_mn = df_cost1[df_cost1['MPO'] == 'SACOG'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
sacog_mn['State'] = 'CA'
sacog_mn['Region'] = 'SACOG Median'
sacog_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

mtc_mn = df_cost1[df_cost1['MPO'] == 'MTC'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
mtc_mn['State'] = 'CA'
mtc_mn['Region'] = 'MTC Median'
mtc_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

scag_mn = df_cost1[df_cost1['MPO'] == 'SCAG'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
scag_mn['State'] = 'CA'
scag_mn['Region'] = 'SCAG Median'
scag_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

# Combine all median dfs, and clean cols
df_cost1 = pd.concat([df_cost1, sacog_mn, mtc_mn, scag_mn])
df_cost1 = df_cost1.sort_values(by=['State', 'MPO', 'Region', 'date_'], ascending = [True, True, True, False])
df_cost1 = df_cost1[['State', 'MPO', 'Region', 'date_', 'Price']]
df_cost1 = df_cost1.reset_index(drop = True)
df_cost1['date_'] = df_cost1['date_'].astype('str')

df_cost1_yr = df_cost1.copy()

df_cost1_yr['date_'] = pd.to_datetime(df_cost1_yr['date_'])
df_cost1_yr['Year'] = df_cost1_yr['date_'].dt.year
df_cost1_yr = df_cost1_yr.drop('date_', axis = 1)
df_cost1_yr = df_cost1_yr.groupby(['State', 'MPO', 'Region', 'Year'], as_index = False)['Price'].mean()
df_cost1_yr = df_cost1_yr.sort_values(by=['State', 'MPO', 'Region', 'Year'], ascending = [True, True, True, False])


# # Export
# path_xlsx = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Cost', 'Cost_1 Sales Price')
# workbook_name = f"{indicator_name} MPO Zillow.xlsx"
# with pd.ExcelWriter(os.path.join(path_xlsx, workbook_name), engine='xlsxwriter') as writer:
#     df_about   .to_excel(writer, index = False, sheet_name = 'About'  , header=False)
#     df_cost1   .to_excel(writer, index = False, sheet_name = 'Monthly'              )
#     df_cost1_yr.to_excel(writer, index = False, sheet_name = 'Annual'               )


display(df_cost1_yr.head(6))

In [ ]:

df_plot = df_cost1.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])

x = 'date_'
y = 'price'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Sale Price by MPO')

    
fig.show()

In [ ]:

df_q = df_cost1.copy()

df_q['date_'] = pd.to_datetime(df_q['date_'])
df_q['Year'] = df_q['date_'].dt.year
df_q['Month'] = df_q['date_'].dt.month
df_q

conditions = [
    df_q['Month'].isin([1, 2, 3])
    , df_q['Month'].isin([4, 5, 6])
    , df_q['Month'].isin([7, 8, 9])
    , df_q['Month'].isin([10, 11, 12])
]

choices = ['Q1', 'Q2', 'Q3', 'Q4']

df_q['Quarter'] = np.select(conditions, choices)

df_q['Year_Q'] = df_q['Year'].astype(str) + '-' + df_q['Quarter'].astype(str)
df_q = df_q.drop('date_', axis = 1)
df_q = df_q.groupby(['State', 'MPO', 'Region', 'Year_Q', 'Quarter'], as_index = False)['Price'].mean()
df_q = df_q.sort_values(by=['State', 'MPO', 'Region', 'Year_Q'], ascending = [True, True, True, True])

display(df_q)

df_plot = df_q.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])

x = 'year_q'
y = 'price'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Sale Price by MPO by Quarter')

fig.show()



In [ ]:

df_gr = df_cost1.copy()

df_gr['date_'] = pd.to_datetime(df_gr['date_'])
df_gr['Year']  = df_gr['date_'].dt.year
df_gr['Month'] = df_gr['date_'].dt.month
df_gr = df_gr.dropna()

df_gr = df_gr.sort_values(['State', 'MPO', 'Region', 'date_'], ascending = [True, True, True, True])
df_gr['Price_GR'] = df_gr['Price'].pct_change()*100
df_gr.loc[df_gr['date_'] == '2000-01-31', 'Price_GR'] = np.nan
df_gr.loc[df_gr['Price_GR'] == np.inf, 'Price_GR'] = np.nan
df_gr = df_gr.sort_values(['Region', 'date_'], ascending = [True, False])
df_gr['first_date'] = df_gr.groupby(['State', 'MPO', 'Region'], as_index = False)['date_'].transform('min')

df_gr = df_gr[df_gr['date_'] != df_gr['first_date']]
df_gr = df_gr[~df_gr['Price_GR'].isna()]
df_gr = df_gr[~df_gr['Price'].isna()]
df_gr = df_gr.reset_index(drop = True)
display(df_gr)


df_gr = df_gr.groupby(['State', 'MPO', 'Region', 'Month'], as_index = False)['Price_GR'].mean()
df_gr = df_gr.sort_values(by=['State', 'MPO', 'Region', 'Month'], ascending = [True, True, True, True])


df_plot = df_gr.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])
display(df_plot)

x = 'month'
y = 'price_gr'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Sale Price Growth Rate by MPO by Month')
    
fig.show()


In [ ]:
df_grq = df_cost1.copy()

df_grq['date_'] = pd.to_datetime(df_grq['date_'])
df_grq['Year']  = df_grq['date_'].dt.year
df_grq['Month'] = df_grq['date_'].dt.month
df_grq = df_grq.dropna()

df_grq = df_grq.sort_values(['State', 'MPO', 'Region', 'date_'], ascending = [True, True, True, True])
df_grq['Price_GR'] = df_grq['Price'].pct_change()*100
df_grq.loc[df_grq['date_'] == '2000-01-31', 'Price_GR'] = np.nan
df_grq.loc[df_grq['Price_GR'] == np.inf, 'Price_GR'] = np.nan
df_grq = df_grq.sort_values(['Region', 'date_'], ascending = [True, False])
df_grq['first_date'] = df_grq.groupby(['State', 'MPO', 'Region'], as_index = False)['date_'].transform('min')

df_grq = df_grq[df_grq['date_'] != df_grq['first_date']]
df_grq = df_grq[~df_grq['Price_GR'].isna()]
df_grq = df_grq[~df_grq['Price'].isna()]
df_grq = df_grq.reset_index(drop = True)

conditions = [
    df_grq['Month'].isin([1, 2, 3])
    , df_grq['Month'].isin([4, 5, 6])
    , df_grq['Month'].isin([7, 8, 9])
    , df_grq['Month'].isin([10, 11, 12])
]

choices = ['Q1', 'Q2', 'Q3', 'Q4']

df_grq['Quarter'] = np.select(conditions, choices)

display(df_grq)


df_grq = df_grq.groupby(['State', 'MPO', 'Region', 'Quarter'], as_index = False)['Price_GR'].mean()
df_grq = df_grq.sort_values(by=['State', 'MPO', 'Region', 'Quarter'], ascending = [True, True, True, True])


df_plot = df_grq.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])
display(df_plot)

x = 'quarter'
y = 'price_gr'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Sale Price Growth Rate by MPO by Quarter')
    
fig.show()


***

Cost_2

***

In [ ]:
indicator_name = 'Cost_2'

df_about = write_about(sample_type=sample_type
                       , indicator_name=indicator_name
                       , year_start=year_start
                       , year_end=year_end
                       , path_config0=path_config0
                       , geography=geography)
print("About page documentation table:")
display(df_about)
print('')


# Read in data
df_cost2 = pd.read_csv(os.path.join(path_raw, "Metro_zori_uc_sfrcondomfr_sm_month.csv")) #nolint
df_cost2 = df_cost2[df_cost2['RegionType'] == "msa"]

# Melt data down
df_cost2 = pd.melt(df_cost2, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'])

# Keep important cols and rename
df_cost2 = df_cost2[['RegionName', 'StateName', 'variable', 'value']]
df_cost2.columns = ["Region", "State", "date_", "Price"]

# Subset data 
df_cost2 = df_cost2[df_cost2['Region'].isin(sacog_state + ca_peers + other_peers)]

# Mutate function to make the MPO col
df_cost2['MPO'] = np.where(df_cost2['Region'].str.contains('|'.join(sacog)), 'SACOG',
                  np.where(df_cost2['Region'].str.contains('|'.join(mtc  )), 'MTC'  ,
                  np.where(df_cost2['Region'].str.contains('|'.join(scag )), 'SCAG' ,
                  np.where(df_cost2['Region'] == 'San Diego', 'SANDAG', df_cost2['Region']))))

# This is similar to before. We want to get the median price and subset accordingly based on MPO.
sacog_mn = df_cost2[df_cost2['MPO'] == 'SACOG'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
sacog_mn['State'] = 'CA'
sacog_mn['Region'] = 'SACOG Median'
sacog_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

mtc_mn = df_cost2[df_cost2['MPO'] == 'MTC'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
mtc_mn['State'] = 'CA'
mtc_mn['Region'] = 'MTC Median'
mtc_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

scag_mn = df_cost2[df_cost2['MPO'] == 'SCAG'].groupby(['MPO', 'date_'])['Price'].median().reset_index()
scag_mn['State'] = 'CA'
scag_mn['Region'] = 'SCAG Median'
scag_mn.columns = ['MPO', 'date_', 'Price', 'State', 'Region']

# Concatenate data
df_cost2 = pd.concat([df_cost2, sacog_mn, mtc_mn, scag_mn], ignore_index=True)

# Sort data
df_cost2 = df_cost2.sort_values(by=['State', 'MPO', 'Region', 'date_'], ascending = [True, True, True, False])
df_cost2 = df_cost2[['State', 'MPO', 'Region', 'date_', 'Price']]
df_cost2 = df_cost2.reset_index(drop = True)
df_cost2['date_'] = df_cost2['date_'].astype('str')


df_cost2_yr = df_cost2.copy()

df_cost2_yr['date_'] = pd.to_datetime(df_cost2_yr['date_'])
df_cost2_yr['Year'] = df_cost2_yr['date_'].dt.year
df_cost2_yr = df_cost2_yr.drop('date_', axis = 1)
df_cost2_yr = df_cost2_yr.groupby(['State', 'MPO', 'Region', 'Year'], as_index = False)['Price'].mean()
df_cost2_yr = df_cost2_yr.sort_values(by=['State', 'MPO', 'Region', 'Year'], ascending = [True, True, True, False])


# # Export
# path_xlsx = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Cost', 'Cost_2 Rent Prices')
# workbook_name = f"{indicator_name} MPO Zillow.xlsx"
# with pd.ExcelWriter(os.path.join(path_xlsx, workbook_name), engine='xlsxwriter') as writer:
#     df_about   .to_excel(writer, index = False, sheet_name = 'About'  , header=False)
#     df_cost2   .to_excel(writer, index = False, sheet_name = 'Monthly'              )
#     df_cost2_yr.to_excel(writer, index = False, sheet_name = 'Annual'               )

display(df_cost2_yr.head(6))

In [ ]:
path_plots = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Cost', 'Cost_2 Rent Prices', 'plots')

df_plot = df_cost2.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])

x = 'date_'
y = 'price'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Rent Price by MPO')

fig.show()



df_plot = df_cost2_yr.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])

x = 'year'
y = 'price'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot, x = x, y = y, color = color, markers = True, labels = labels)
fig.update_layout(title = 'Annual Average Home Rent Price by MPO')

fig.show()

In [ ]:

df_q = df_cost2.copy()

df_q['date_'] = pd.to_datetime(df_q['date_'])
df_q['Year'] = df_q['date_'].dt.year
df_q['Month'] = df_q['date_'].dt.month
df_q

conditions = [
    df_q['Month'].isin([1, 2, 3])
    , df_q['Month'].isin([4, 5, 6])
    , df_q['Month'].isin([7, 8, 9])
    , df_q['Month'].isin([10, 11, 12])
]

choices = ['Q1', 'Q2', 'Q3', 'Q4']

df_q['Quarter'] = np.select(conditions, choices)

df_q['Year_Q'] = df_q['Year'].astype(str) + '-' + df_q['Quarter'].astype(str)
df_q = df_q.drop('date_', axis = 1)
df_q = df_q.groupby(['State', 'MPO', 'Region', 'Year_Q', 'Quarter'], as_index = False)['Price'].mean()
df_q = df_q.sort_values(by=['State', 'MPO', 'Region', 'Year_Q'], ascending = [True, True, True, True])

display(df_q)

df_plot = df_q.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])

x = 'year_q'
y = 'price'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Sale Price by MPO by Quarter')

fig.show()



In [ ]:
df_gr = df_cost2.copy()

df_gr['date_'] = pd.to_datetime(df_gr['date_'])
df_gr['Year']  = df_gr['date_'].dt.year
df_gr['Month'] = df_gr['date_'].dt.month
df_gr = df_gr.dropna()

df_gr = df_gr.sort_values(['State', 'MPO', 'Region', 'date_'], ascending = [True, True, True, True])
df_gr['Price_GR'] = df_gr['Price'].pct_change()*100
df_gr.loc[df_gr['date_'] == '2000-01-31', 'Price_GR'] = np.nan
df_gr.loc[df_gr['Price_GR'] == np.inf, 'Price_GR'] = np.nan
df_gr = df_gr.sort_values(['Region', 'date_'], ascending = [True, False])
df_gr['first_date'] = df_gr.groupby(['State', 'MPO', 'Region'], as_index = False)['date_'].transform('min')

df_gr = df_gr[df_gr['date_'] != df_gr['first_date']]
df_gr = df_gr[~df_gr['Price_GR'].isna()]
df_gr = df_gr[~df_gr['Price'].isna()]
df_gr = df_gr.reset_index(drop = True)
display(df_gr)


df_gr = df_gr.groupby(['State', 'MPO', 'Region', 'Month'], as_index = False)['Price_GR'].mean()
df_gr = df_gr.sort_values(by=['State', 'MPO', 'Region', 'Month'], ascending = [True, True, True, True])


df_plot = df_gr.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])
display(df_plot)

x = 'month'
y = 'price_gr'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Sale Price Growth Rate by MPO by Month')
    
fig.show()


In [ ]:
df_grq = df_cost2.copy()

df_grq['date_'] = pd.to_datetime(df_grq['date_'])
df_grq['Year']  = df_grq['date_'].dt.year
df_grq['Month'] = df_grq['date_'].dt.month
df_grq = df_grq.dropna()

df_grq = df_grq.sort_values(['State', 'MPO', 'Region', 'date_'], ascending = [True, True, True, True])
df_grq['Price_GR'] = df_grq['Price'].pct_change()*100
df_grq.loc[df_grq['date_'] == '2000-01-31', 'Price_GR'] = np.nan
df_grq.loc[df_grq['Price_GR'] == np.inf, 'Price_GR'] = np.nan
df_grq = df_grq.sort_values(['Region', 'date_'], ascending = [True, False])
df_grq['first_date'] = df_grq.groupby(['State', 'MPO', 'Region'], as_index = False)['date_'].transform('min')

df_grq = df_grq[df_grq['date_'] != df_grq['first_date']]
df_grq = df_grq[~df_grq['Price_GR'].isna()]
df_grq = df_grq[~df_grq['Price'].isna()]
df_grq = df_grq.reset_index(drop = True)

conditions = [
    df_grq['Month'].isin([1, 2, 3])
    , df_grq['Month'].isin([4, 5, 6])
    , df_grq['Month'].isin([7, 8, 9])
    , df_grq['Month'].isin([10, 11, 12])
]

choices = ['Q1', 'Q2', 'Q3', 'Q4']

df_grq['Quarter'] = np.select(conditions, choices)

display(df_grq)


df_grq = df_grq.groupby(['State', 'MPO', 'Region', 'Quarter'], as_index = False)['Price_GR'].mean()
df_grq = df_grq.sort_values(by=['State', 'MPO', 'Region', 'Quarter'], ascending = [True, True, True, True])


df_plot = df_grq.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

df_plot1 = df_plot[ df_plot['region'].str.contains('Median')]
df_plot2 = df_plot[~df_plot['region'].str.contains('Median')]
df_plot2 = df_plot2[~df_plot2['mpo'].isin(['MTC', 'SACOG', 'SCAG'])]
df_plot = pd.concat([df_plot1, df_plot2])
display(df_plot)

x = 'quarter'
y = 'price_gr'
color = 'mpo'
labels = 'mpo'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Sale Price Growth Rate by MPO by Quarter')
    
fig.show()
